# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">7주차 · 추천 파이프라인 — 데이터 점검부터 결과 파일까지 실행 한 번으로</mark>

지난 여섯 주 동안 추천은 **셀을 하나씩 눌러서** 만들었습니다.

실제 금융 회사는 고객 수만 명의 추천을 **매일 밤 컴퓨터가 알아서** 만듭니다. 사람이 매번 셀을 누를 수 없습니다.

오늘은 6주차의 최종 추천(행렬분해 + 위험도 상한)을 **그대로 두고**, 그 추천을 만드는 다섯 단계를 **실행 한 번**으로 잇습니다. 이렇게 단계를 이어 붙인 것을 **파이프라인(pipeline)** 이라고 합니다.

| 단계 | 하는 일 |
|---|---|
| ① 데이터 점검 | 들어온 데이터에 빈 값 · 중복 · 없는 번호가 없는지 검사한다 |
| ② 학습 | 거래 기록 전체로 행렬분해를 학습한다 |
| ③ 일괄 추천 | 300명 × 10개를 한꺼번에 계산한다 (위험도 상한 포함) |
| ④ 규격 검사 | 결과 표가 정해 둔 약속을 지켰는지 검사한다 |
| ⑤ 저장 | CSV 파일 하나로 저장한다 → 다음 주 API 가 읽는다 |

다섯 단계는 모두 **`recommend_batch.py` 파일 하나**에 들어 있습니다. 이 노트북은 그 파일을 **읽고 → 실행하고 → 결과를 확인**합니다.

**오늘의 결론을 미리 말하면** — 점수는 6주차와 같은 **0.3899** 입니다. 추천은 그대로이고, 만드는 데 **사람 손이 빠집니다.**

---

### 오늘의 구성 — 50분 (PPT + 노트북)

| 시간 | 어디서 | 하는 일 |
|---|---|---|
| 개념 30분 | PPT | 파이프라인 · 데이터 점검 · 학습과 서빙 · 일괄 추천 · 결과 파일 규격 · 재현성 |
| 실습 20분 | **이 노트북** | 맨 위부터 실행 → 7.1 데이터 점검 → 7.2 스크립트 읽기 → 7.3 실행 → 7.4 결과 파일 → 7.5 채점 → 🗣 함께 보기 |

### 이 노트북 읽는 법 — 🎯 와 📖

- **🎯 수업** — 수업 시간에 셀을 실행하고 **결과를 함께 읽는** 절입니다
  - 수업에서는 설명 글을 다 읽지 않습니다. 셀 실행과 「결과 읽는 법」만 봅니다
- **📖 혼자 읽기** — 수업에서는 건너뛰고 **복습할 때 읽는** 부분입니다
  - 절 전체가 📖 인 곳(7.6 · 7.7 · 7.8)과, 🎯 절 안의 「왜 배우나 · 배경 지식 · 어떻게 동작하나」 블록이 여기에 속합니다

⏱ **실습을 시작하면 먼저 메뉴의 `런타임 → 모두 실행` 을 눌러 두세요.** 전체 실행에 10초쯤 걸립니다.

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기 · 이 노트북을 여는 법</mark>

1. https://colab.research.google.com/github/welovecherry/recsys/blob/main/notebooks/07_batch_pipeline.ipynb
2. 구글 계정으로 로그인합니다.
3. **경고창이 뜨면 `Run anyway` 를 누릅니다.**
4. **아래 셀의 ▶ 버튼을 누릅니다.** 실습 자료를 받아옵니다. 10초쯤 걸립니다.
5. 그다음부터는 위에서 아래로 셀을 하나씩 실행하면 됩니다.

> ⚠ **고친 내용을 남기려면** 메뉴에서 `파일 → 드라이브에 사본 저장` 을 눌러 주세요.

---

**이 노트북이 쓰는 것**

| 무엇 | 쓰임 |
|---|---|
| `data/items.csv` · `users.csv` · `interactions.csv` | 종목 100개 · 투자자 300명 · 거래 기록 8,086건 |
| `recsys.py` | 데이터 읽기 · 시간순 분할 · 채점 · 사용자-아이템 행렬 |
| `recommend_batch.py` | **오늘의 산출물** — 데이터 점검부터 저장까지 실행 한 번으로 하는 스크립트 |

**이 노트북이 만드는 것**

| 무엇 | 어디에 |
|---|---|
| 300명 × 추천 10개 결과 파일 | `output/recommendations.csv` |

⚠ **지난주 노트북을 먼저 돌릴 필요가 없습니다.** 이 노트북 하나로 끝납니다.

⚠ **따로 설치할 것이 없습니다.** 오늘 쓰는 도구는 Colab 에 처음부터 들어 있습니다.

---

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">먼저 이 셀부터 실행하세요  ▶</mark>

In [1]:
# 이 셀에서 하는 일 — Colab 에서 열었으면 실습 자료를 내려받는다
# 왜 하나 — Colab 은 열 때마다 빈 컴퓨터라 데이터·recsys.py 가 없다. 내 컴퓨터면 그냥 넘어간다

import os          # 폴더를 만들고 옮겨 다니는 도구
import sys         # 지금 파이썬이 어떤 환경인지 알려 주는 도구
import subprocess  # 터미널 명령을 파이썬에서 대신 실행해 주는 도구

if "google.colab" in sys.modules:                    # Colab 이면 이 안이 실행된다
    if os.path.exists("/content/recsys"):            # 전에 받아 둔 것이 있으면 최신으로
        subprocess.run(["git", "-C", "/content/recsys", "pull", "-q", "--ff-only"])
    else:                                            # 처음이면 통째로 내려받는다
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/welovecherry/recsys.git", "/content/recsys"])
    os.chdir("/content/recsys/notebooks")            # 노트북 폴더 안으로 이동
    print("준비 끝 —", os.getcwd(), "· 아래 셀부터 차례로 실행하세요.")
else:
    print("내 컴퓨터에서 실행 중입니다 —", os.getcwd(), "· 따로 받아올 것이 없습니다.")

내 컴퓨터에서 실행 중입니다 — /Users/hong/workspaces/org_physical-spark/course-recsys/notebooks · 따로 받아올 것이 없습니다.


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">오늘 나오는 말 여덟 개</mark>

**1번은 오늘의 큰 그림, 2\~6번은 파이프라인의 단계, 7\~8번은 파이프라인을 믿고 쓰기 위한 성질입니다.**

여기서 다 외우지 않아도 됩니다. 괄호 안의 절에서 예와 함께 다시 설명합니다.

1. **파이프라인 (pipeline)** — 여러 단계를 **순서대로 이어 붙여 한 번에 돌리는 것** (맨 위 표)
   - 앞 단계의 결과가 다음 단계의 입력이 됩니다
   - 오늘의 파이프라인 — 데이터 점검 → 학습 → 일괄 추천 → 규격 검사 → 저장

2. **데이터 점검 (data validation)** — 학습하기 **전에** 들어온 데이터가 깨지지 않았는지 검사하는 것 (7.1)
   - 노션 계획표의 「데이터 정리」에 해당합니다
   - 빈 값 · 중복 · 목록에 없는 번호 같은 것을 셉니다

3. **학습 (training) · 서빙 (serving)** — 모델을 만드는 일과, 만든 모델로 추천을 **꺼내 주는** 일 (7.6)
   - 학습은 무겁고 가끔(하루 한 번), 서빙은 가볍고 자주(고객이 앱을 열 때마다)
   - 둘을 나누면 서빙할 때 다시 학습할 필요가 없습니다

4. **일괄 추천 (batch recommendation)** — 모든 고객의 추천을 **한꺼번에 미리** 계산해 두는 방식 (7.3)
   - 반대는 **실시간 추천(real-time recommendation)** — 고객이 요청할 때 그 자리에서 계산
   - 우리는 일괄 추천입니다. 다음 주 API 는 미리 만든 결과를 읽기만 합니다

5. **결과 파일 규격 (output schema)** — 결과 파일의 **열 이름 · 순서 · 개수**를 정해 둔 약속 (7.4)
   - 만드는 쪽(오늘 스크립트)과 읽는 쪽(다음 주 API)이 같은 약속을 지켜야 파일이 맞물립니다

6. **규격 검사** — 결과 표가 규격을 지켰는지 **코드가 자동으로** 검사하는 것 (7.4)
   - 데이터 점검이 「들어오는 데이터」 검사라면, 규격 검사는 「나가는 데이터」 검사입니다

7. **스크립트 (script)** — 위에서 아래로 **한 번에 실행되는** 파이썬 파일(`.py`) (7.2 \~ 7.3)
   - 노트북(`.ipynb`)은 사람이 셀을 눌러야 돌아가고, 스크립트는 명령 한 줄이면 끝까지 돌아갑니다

8. **재현성 (reproducibility)** — 같은 데이터로 다시 돌리면 **같은 결과**가 나오는 성질 (7.7)
   - `random_state=42` 처럼 무작위 요소를 고정해서 지킵니다

---

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 준비 — 데이터 읽기와 시간순 분할</mark>

**여기서 하는 일** — 데이터를 읽고, 3주차부터 쓰던 **시간순 분할**로 나눕니다. 그리고 오늘의 스크립트 `recommend_batch.py` 를 불러옵니다.

- 시간순 분할은 7.5 에서 채점할 때만 씁니다. 그래야 6주차 **0.3899** 와 비교할 수 있습니다.
- `import recommend_batch` — 스크립트 파일 안의 함수를 노트북에서도 꺼내 쓸 수 있게 불러옵니다.

In [2]:
# 이 셀에서 하는 일 — 도구를 불러오고, 데이터를 읽고, 시간순으로 나눈다
# 왜 하나 — 7.1 에서 데이터를 점검하고, 7.5 에서 6주차와 같은 기준으로 채점하려면 필요하다

import sys                                  # 파이썬이 파일을 찾는 경로를 다루는 도구
sys.path.insert(0, ".")                     # 지금 폴더에서 recsys.py · recommend_batch.py 를 찾게 한다
sys.path.insert(0, "notebooks")             # 한 단계 안쪽 폴더(notebooks)도 찾게 한다

import pandas as pd                         # 표를 다루는 도구. pd 라고 부른다
import recsys                               # 이 수업용 도구 모음 — 읽기 · 분할 · 채점
import recommend_batch                      # 오늘의 스크립트 — 안에 든 함수를 꺼내 쓸 수 있다

items, users, interactions = recsys.load()                  # 종목 · 투자자 · 거래 기록 세 표
train, test, 기준시점 = recsys.split_by_time(interactions)   # 3주차부터 쓰던 시간순 분할

print(f"종목 {len(items)}개 · 투자자 {len(users)}명 · 거래 기록 {len(interactions):,}건")
print(f"채점용 — 학습 구간 {len(train):,}건 · 채점 구간 {len(test):,}건")

종목 100개 · 투자자 300명 · 거래 기록 8,086건
채점용 — 학습 구간 7,095건 · 채점 구간 991건


**결과 읽는 법**

- 첫 줄은 오늘 쓰는 데이터 세 가지의 크기입니다 — 종목 **100개** · 투자자 **300명** · 거래 기록 **8,086건**.
- 둘째 줄은 채점용으로 자른 결과입니다.
  - **학습 구간 7,095건**으로 배우고 **채점 구간 991건**(마지막 1개월)으로 맞혔는지 봅니다. 6주차와 같습니다.
  - 스크립트가 만드는 **서비스용** 추천은 자르지 않은 **8,086건 전체**로 학습합니다. 7.3 에서 봅니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 7.1 데이터 점검 — 학습하기 전에 들어온 데이터를 검사한다</mark>

**7.1 에서 하는 일** — 거래 기록 · 종목 · 투자자 세 표에 **문제가 있는 줄이 몇 개인지** 여섯 가지로 셉니다. 모두 0 이면 통과입니다.

---
> 📖 **여기부터 「자세히」 전까지는 복습할 때 읽는 부분입니다.** 수업에서는 건너뜁니다.

**왜 배우나**
- 노션 계획표의 7주차 한 줄은 「**데이터 정리부터** 배치 추천 · 결과 저장까지」입니다. 파이프라인의 첫 단계가 데이터 점검입니다.
- 지금까지는 데이터가 깨끗하다고 **믿고** 썼습니다. 매일 밤 자동으로 돌리면 사람이 데이터를 볼 기회가 없습니다.

**왜 중요한가**
- 실제 거래 기록은 여러 시스템에서 모여서 자주 깨집니다.
  - 전송 오류로 같은 거래가 두 번 들어오기도 하고, 상장 폐지된 종목 번호가 남아 있기도 합니다.
- 깨진 데이터로 학습하면 **오류 없이 조용히 틀린 추천**이 나옵니다. 이것이 가장 위험합니다.
  - 예 — 같은 거래가 두 번 들어오면 그 종목이 실제보다 인기 있게 보여서 추천에 더 자주 나옵니다.
- 그래서 실무 파이프라인은 학습 **앞에서** 먼저 검사하고, 문제가 있으면 **멈춥니다.**

**알아 둘 배경 지식**
- **빈 값 (missing value)** — 있어야 할 칸이 비어 있는 것. pandas 에서는 `NaN` 으로 보입니다
- **중복 (duplicate)** — 모든 열이 똑같은 줄이 두 번 이상 있는 것
- **참조 무결성 (referential integrity)** — 거래 기록에 적힌 종목 번호 · 투자자 번호가 **목록에 실제로 있는가**
  - 거래 기록에 `I999` 가 있는데 종목 목록에 `I999` 가 없으면, 그 종목은 이름도 위험도도 알 수 없습니다
- **허용 범위** — 값이 정해진 범위 안에 있는가. 투자성향은 세 가지, 위험도는 1 \~ 5

**어떻게 동작하나**
1. 세 표의 빈 칸을 모두 센다
2. 거래 기록에서 모든 열이 같은 줄을 센다
3. 거래 기록의 종목 번호가 종목 목록에 없는 줄을 센다
4. 거래 기록의 투자자 번호가 투자자 목록에 없는 줄을 센다
5. 투자성향이 안정형 · 성장형 · 공격형이 아닌 투자자를 센다
6. 위험도가 1 \~ 5 를 벗어난 종목을 센다
- 여섯 개가 모두 0 이면 통과, 하나라도 0 이 아니면 스크립트는 **학습을 시작하지 않고 멈춥니다.**

---

#### 자세히

- 이 검사는 `recommend_batch.py` 안의 `데이터_점검` 함수가 합니다. 아래 셀은 그 함수를 불러서 결과만 표로 봅니다.
- 날짜가 어느 기간인지도 함께 봅니다. 기간이 예상과 다르면(예: 미래 날짜) 데이터를 의심해야 합니다.

In [3]:
# 이 셀에서 하는 일 — 스크립트의 데이터 점검 함수를 불러 여섯 가지 검사 결과를 표로 본다
# 왜 하나 — 학습 전에 데이터가 깨지지 않았는지 확인한다. 모두 0 이어야 통과다

점검 = recommend_batch.데이터_점검(items, users, interactions)   # {검사 이름: 문제 개수}
점검_표 = pd.DataFrame({"검사": list(점검.keys()), "문제 개수": list(점검.values())})
print(점검_표.to_string(index=False))
print(f"\n거래 기록 날짜 — {interactions['ts'].min().date()} 부터 {interactions['ts'].max().date()} 까지")
print("→ 통과" if sum(점검.values()) == 0 else "→ 실패 — 학습하지 않고 멈춰야 한다")

               검사  문제 개수
              빈 값      0
        똑같은 거래 기록      0
  종목 목록에 없는 종목 번호      0
투자자 목록에 없는 투자자 번호      0
    세 가지가 아닌 투자성향      0
  1 ~ 5 를 벗어난 위험도      0

거래 기록 날짜 — 2025-08-31 부터 2026-08-30 까지
→ 통과


**결과 읽는 법**

- 표의 한 줄이 검사 하나입니다. **여섯 가지 모두 0** 입니다.
  - 빈 칸도, 두 번 들어온 거래도, 목록에 없는 번호도, 범위를 벗어난 값도 없습니다.
- 날짜는 **2025-08-31 부터 2026-08-30 까지**, 1년치입니다. 미래 날짜는 없습니다.
- 마지막 줄 **→ 통과** — 이 데이터로 학습해도 됩니다.
- 우리 데이터는 수업용으로 만든 데이터라 깨끗합니다. 실제 데이터에서는 이 표에 0 이 아닌 숫자가 자주 나옵니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 7.2 스크립트 읽기 — 다섯 단계가 어디 있는지 본다</mark>

**7.2 에서 하는 일** — 실행하기 **전에** `recommend_batch.py` 의 `main` 함수를 화면에 띄워, 다섯 단계가 몇째 줄에 있는지 읽습니다.

---
> 📖 **여기부터 「자세히」 전까지는 복습할 때 읽는 부분입니다.** 수업에서는 건너뜁니다.

**왜 배우나**
- 6주차까지 셀 수십 개에 흩어져 있던 일이 이 파일 하나에 모여 있습니다. **무엇이 어디로 갔는지** 알아야 나중에 고칠 수 있습니다.
- 남이 만든 스크립트를 실행하기 전에 **무엇을 하는지 먼저 읽는 습관**은 실무의 기본입니다.

**왜 중요한가**
- 스크립트는 사람이 지켜보지 않을 때 돌아갑니다. 무엇을 하는지 모르는 파일을 매일 밤 돌리면 문제가 생겨도 알아채지 못합니다.
- `main` 함수만 읽어도 전체 흐름이 보이게 만드는 것이 좋은 스크립트입니다. 세부 계산은 다른 함수로 빼 두었습니다.

**알아 둘 배경 지식**
- **`main` 함수** — 스크립트를 실행하면 처음으로 불리는 함수. 「이 파일이 하는 일의 목차」입니다
- **함수 (function)** — 이름을 붙여 둔 계산 묶음. `학습하기(...)` 처럼 이름으로 부릅니다 (6주차의 `점수순으로_종목_정렬` 과 같은 것)
- **`raise SystemExit(...)`** — 「여기서 멈춰라」. 문제가 있으면 다음 단계로 가지 않고 메시지를 남기고 끝냅니다
- **왜 행렬분해로 묶나** — 6주차에서 네 모델을 비교했을 때 행렬분해(0.3918)가 FM(0.3463) · MLP(0.3283)보다 높았습니다. 그래서 오늘 묶는 모델은 행렬분해 + 위험도 상한입니다

**어떻게 동작하나 — `main` 의 순서**
1. `recsys.load()` — 세 표를 읽는다
2. `데이터_점검(...)` — 하나라도 0 이 아니면 **멈춘다** (①)
3. `학습하기(interactions, items)` — **거래 기록 전체**로 학습한다 (②)
4. `일괄_추천(...)` — 300명 × 10개 표를 만든다 (③)
5. `규격_검사(...)` — 문제가 있으면 **멈춘다** (④)
6. `표.to_csv(...)` — 파일로 저장한다 (⑤)
- 저장이 **검사 뒤**에 있다는 점이 중요합니다. 검사를 통과하지 못한 파일은 아예 만들어지지 않습니다.

---

#### 자세히

- `inspect.getsource` — 함수의 코드를 글자 그대로 꺼내 보여 주는 도구입니다. 파일을 직접 열어도 되지만, 이렇게 하면 필요한 함수만 볼 수 있습니다.
- Colab 이라면 왼쪽 📁 아이콘 → `notebooks/recommend_batch.py` 를 두 번 눌러 파일 전체를 열어 볼 수도 있습니다.

In [4]:
# 이 셀에서 하는 일 — 스크립트의 main 함수를 글자 그대로 화면에 띄운다
# 왜 하나 — 실행하기 전에 다섯 단계가 어떤 순서로 있는지 읽는다

import inspect                                     # 함수의 코드를 꺼내 보여 주는 도구

print(inspect.getsource(recommend_batch.main))     # getsource = 함수의 코드를 글자로

def main():
    items, users, interactions = recsys.load()

    점검 = 데이터_점검(items, users, interactions)
    if sum(점검.values()) > 0:
        raise SystemExit(f"① 데이터 점검 실패 — {점검}")
    print(f"① 데이터 점검 — 여섯 가지 모두 0건 · 통과")

    묶음 = 학습하기(interactions, items)                    # 서비스용은 거래 기록 전체로 학습
    print(f"② 학습 — 거래 기록 {len(interactions):,}건 · 투자자 {len(묶음['사용자_목록'])}명")

    표 = 일괄_추천(묶음, users, items)
    print(f"③ 일괄 추천 — {len(표):,}줄")

    문제 = 규격_검사(표, users, 묶음)
    if 문제:
        raise SystemExit("④ 규격 검사 실패 — " + " / ".join(문제[:5]))
    print("④ 규격 검사 — 문제 0개 · 통과")

    출력_파일.parent.mkdir(exist_ok=True)
    표.to_csv(출력_파일, index=False, encoding="utf-8-sig")  # utf-8-sig = 엑셀에서 한글이 안 깨진다
    print(f"⑤ 저장 — {출력_파일} · {표['user_id'].nunique()}명 × 10개")



**결과 읽는 법**

- `main` 함수의 코드 전체입니다. 위에서 아래로 읽으면 **①부터 ⑤까지** 차례로 나옵니다.
  - `print(f"① …")` 줄이 단계마다 하나씩 있습니다. 실행하면 이 줄들이 화면에 찍혀서 어디까지 왔는지 알려 줍니다.
- `if sum(점검.values()) > 0:` 아래의 `raise SystemExit` — 데이터 점검에서 문제가 나오면 여기서 멈춥니다.
- `if 문제:` 아래의 `raise SystemExit` — 규격 검사에서 문제가 나오면 저장하기 **전에** 멈춥니다.
- `학습하기(interactions, items)` — 채점용 `train` 이 아니라 **전체 `interactions`** 로 학습합니다. 서비스용이기 때문입니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 7.3 스크립트 실행 — 명령 한 줄로 다섯 단계를 끝까지</mark>

**7.3 에서 하는 일** — `!python recommend_batch.py` **한 줄**로 스크립트를 실행합니다. 노트북 셀을 하나씩 누르지 않아도 데이터 점검부터 저장까지 끝납니다.

---
> 📖 **여기부터 「자세히」 전까지는 복습할 때 읽는 부분입니다.** 수업에서는 건너뜁니다.

**왜 배우나**
- 오늘 수업의 핵심 장면입니다. 6주차까지 셀 수십 개로 하던 일이 **명령 한 줄**이 됩니다.

**왜 중요한가**
- 명령 한 줄로 돌아가면, 사람 대신 **컴퓨터가 정해진 시각에 실행**할 수 있습니다.
  - 매일 새벽 2시에 실행하도록 걸어 두는 것을 **예약 실행(스케줄링, scheduling)** 이라고 합니다
  - 그러면 고객은 아침마다 전날까지의 거래가 반영된 추천을 받습니다
- 사람이 셀을 누르면 순서를 틀리거나 한 셀을 빼먹을 수 있습니다. 스크립트는 **늘 같은 순서로** 돕니다.

**알아 둘 배경 지식**
- **`.ipynb` 와 `.py`**
  - `.ipynb` (노트북) — 셀 · 설명 글 · 실행 결과가 함께 저장된 파일. 사람이 읽고 실험하기 좋다
  - `.py` (스크립트) — 파이썬 코드만 있는 파일. 처음부터 끝까지 한 번에 실행된다
- **셀 맨 앞의 `!`** — 「이 줄은 파이썬 코드가 아니라 **컴퓨터에 내리는 명령**이다」라는 표시 (Colab · 주피터에서만)
  - `python recommend_batch.py` = 「파이썬으로 recommend_batch.py 파일을 실행하라」
- **경로 (path)** — 파일이 있는 자리. `output/recommendations.csv` = 지금 폴더 안의 `output` 폴더 안의 `recommendations.csv`
- **서비스용은 왜 전체 기간으로 학습하나**
  - 채점할 때는 마지막 한 달을 **숨겨야** 실력을 정직하게 잴 수 있습니다 (3주차)
  - 서비스용 추천은 **내일 고객에게 보여 줄** 것이라 가장 최근 기록까지 모두 써야 합니다

**어떻게 동작하나**
1. 컴퓨터가 새 파이썬을 하나 띄워 `recommend_batch.py` 를 맨 위부터 읽는다
2. 맨 아래 `if __name__ == "__main__": main()` 에 닿으면 `main` 을 부른다
3. `main` 이 ① \~ ⑤ 를 차례로 하고, 단계마다 한 줄씩 화면에 찍는다
4. 끝나면 `output/recommendations.csv` 가 남는다

---

#### 자세히

- 이 셀은 노트북 안의 변수(`items` · `train` 등)를 **전혀 쓰지 않습니다.** 스크립트가 데이터를 스스로 읽습니다.
- 그래서 이 셀 하나만 있어도, 위의 셀을 하나도 실행하지 않아도 결과 파일이 만들어집니다.

In [5]:
# 이 셀에서 하는 일 — 스크립트를 실행한다. 맨 앞의 ! 는 「컴퓨터에 내리는 명령」이라는 표시다
# 왜 하나 — 데이터 점검부터 저장까지 명령 한 줄로 끝나는지 확인한다

!python recommend_batch.py

① 데이터 점검 — 여섯 가지 모두 0건 · 통과


② 학습 — 거래 기록 8,086건 · 투자자 300명


③ 일괄 추천 — 3,000줄


④ 규격 검사 — 문제 0개 · 통과
⑤ 저장 — output/recommendations.csv · 300명 × 10개


**결과 읽는 법**

- 스크립트가 단계마다 찍은 다섯 줄입니다. 위에서부터 차례로 끝났습니다.
  - **①** 데이터 점검 — 여섯 가지 모두 0건, 통과
  - **②** 학습 — 거래 기록 **8,086건** 전체, 투자자 **300명**
    - 채점용(학습 구간)은 285명이었습니다. 마지막 한 달에 처음 온 신규 15명이 이제 들어왔습니다
  - **③** 일괄 추천 — 300명 × 10개 = **3,000줄**
  - **④** 규격 검사 — 문제 0개, 통과
  - **⑤** 저장 — `output/recommendations.csv`
- 셀 하나, 명령 한 줄로 끝났습니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 7.4 결과 파일 확인 — 규격대로 나왔나</mark>

**7.4 에서 하는 일** — 스크립트가 만든 파일을 열어 **규격**대로 나왔는지 눈으로 확인합니다. 안정형 U0001 의 10개도 봅니다.

---
> 📖 **여기부터 「자세히」 전까지는 복습할 때 읽는 부분입니다.** 수업에서는 건너뜁니다.

**왜 배우나**
- 이 파일이 다음 주 API 가 읽을 **유일한 입력**입니다. 파일의 모양이 곧 다음 주와의 약속입니다.

**왜 중요한가**
- 다음 주 API 는 이 파일을 **읽기만** 합니다. 열 이름 하나만 달라도 API 가 멈춥니다.
- 그래서 만드는 쪽과 읽는 쪽이 **같은 규격**을 미리 정해 두고, 스크립트의 ④ 규격 검사가 그 약속을 지켰는지 자동으로 확인합니다.
- 금융에서는 이 파일에 **위험도 상한을 넘는 종목이 하나도 없어야** 합니다(6주차 적합성원칙). 규격 검사가 이것도 봅니다.

**알아 둘 배경 지식**
- **CSV (comma-separated values)** — 값을 쉼표로 나눠 적은 글자 파일. 엑셀 · 메모장 · 파이썬 어디서든 열 수 있습니다
- **인코딩 (encoding)** — 글자를 컴퓨터의 숫자로 바꾸는 방식
  - `utf-8-sig` 로 저장하면 엑셀에서 열어도 **한글이 깨지지 않습니다**
- **규격 (이 수업에서 정한 약속)**

| 열 | 뜻 | 예 |
|---|---|---|
| `user_id` | 투자자 번호 | U0001 |
| `rank` | 추천 순위 1 \~ 10 | 1 |
| `item_id` | 종목 번호 | I002 |
| `item_name` | 종목 이름 | TIGER 미국배당다우존스 |
| `risk_level` | 위험도 1 \~ 5 | 2 |
| `persona` | 투자성향 | 안정형 |
| `score` | 행렬분해 점수 (기록이 없으면 0) | 0.6211 |

**어떻게 동작하나 — 규격 검사 다섯 가지 (스크립트 ④)**
1. 열 이름과 순서가 규격과 같은가
2. 300명 모두 추천이 있는가
3. 사람마다 정확히 10줄인가
4. 투자성향의 상한을 넘는 종목이 없는가 (안정형 3 · 성장형 4 · 공격형 5)
5. 이미 담은 종목을 추천하지 않았는가

---

#### 자세히

- 아래 셀은 파일을 **다시 읽어서** 확인합니다. 메모리 안의 표가 아니라 실제로 저장된 파일을 봐야 「다음 주 API 가 읽을 것」을 확인한 것이 됩니다.

In [6]:
# 이 셀에서 하는 일 — 스크립트가 만든 결과 파일을 읽어 크기 · 열 · U0001 의 추천을 본다
# 왜 하나 — 다음 주 API 가 읽을 파일이 규격대로 나왔는지 눈으로 확인한다

결과 = pd.read_csv("output/recommendations.csv", encoding="utf-8-sig")   # 저장된 파일을 다시 읽는다

print(f"크기 {결과.shape}  ← 3,000줄 × 열 7개")
print("열 :", list(결과.columns))
print(f"안정형 투자자에게 간 추천 중 가장 높은 위험도 : {결과[결과['persona'] == '안정형']['risk_level'].max()}")
결과[결과["user_id"] == "U0001"]                     # U0001 의 10줄만

크기 (3000, 7)  ← 3,000줄 × 열 7개
열 : ['user_id', 'rank', 'item_id', 'item_name', 'risk_level', 'persona', 'score']
안정형 투자자에게 간 추천 중 가장 높은 위험도 : 3


,user_id,rank,item_id,item_name,risk_level,persona,score
0,U0001,1,I002,TIGER 미국배당다우존스,2,안정형,0.6211
1,U0001,2,I043,Vanguard FTSE Developed Markets ETF,3,안정형,0.4705
2,U0001,3,I014,KT,2,안정형,0.3974
3,U0001,4,I011,KODEX 단기채권,1,안정형,0.3722
4,U0001,5,I021,Verizon Communications,2,안정형,0.3690
5,U0001,6,I015,LG유플러스,2,안정형,0.3645
6,U0001,7,I033,TIGER 은행고배당플러스TOP10,2,안정형,0.3356
7,U0001,8,I020,Johnson & Johnson,2,안정형,0.3127
8,U0001,9,I035,TIGER 200,3,안정형,0.3036
9,U0001,10,I055,Apple,3,안정형,0.2801


**결과 읽는 법**

- **크기 `(3000, 7)`** — 300명 × 10개 = 3,000줄, 열 7개. 규격과 같습니다.
- **열** — `user_id` 부터 `score` 까지 규격의 이름 · 순서 그대로입니다.
- **안정형에게 간 가장 높은 위험도 3** — 안정형 상한(3)을 넘는 추천이 없습니다.
- 아래 표는 안정형 **U0001** 의 추천 10개입니다.
  - `rank` 가 1 부터 10 까지, `risk_level` 은 모두 3 이하입니다.
  - 6주차 6.2 에서 4주차 모델이 U0001 에게 준 위험도 4 \~ 5 종목(TIGER 2차전지테마 · 크래프톤 등)이 없습니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 7.5 같은 추천인지 채점 — 6주차 0.3899 와 비교</mark>

**7.5 에서 하는 일** — 스크립트 안의 함수로 **채점용**(학습 구간만) 추천을 만들어 Recall@10 을 잽니다. 6주차와 같은 0.3899 가 나와야 합니다.

---
> 📖 **여기부터 「자세히」 전까지는 복습할 때 읽는 부분입니다.** 수업에서는 건너뜁니다.

**왜 배우나**
- 셀을 파일로 옮기면서 **무언가 빠지거나 바뀌지 않았는지** 확인해야 합니다. 같은 기준으로 채점해 같은 점수가 나오면 옮기기가 정확했다는 증거입니다.

**왜 중요한가**
- 코드를 옮기거나 정리하는 일(**리팩터링, refactoring**)은 실무에서 매우 흔합니다.
- 옮긴 뒤 **결과가 그대로인지 숫자로 확인하는 것**이 정리의 마지막 단계입니다. 확인하지 않으면 「정리했더니 추천이 바뀌었다」를 아무도 모릅니다.

**알아 둘 배경 지식**
- **채점용 vs 서비스용**
  - 채점용 — 학습 구간(7,095건)만으로 학습. 마지막 한 달을 숨기고 맞혀 본다
  - 서비스용 — 전체(8,086건)로 학습. 7.3 에서 스크립트가 만든 파일
  - 점수는 **채점용**으로만 잴 수 있습니다. 서비스용은 맞혀 볼 미래가 없습니다
- **Recall@10** (1주차) — 다음 달 실제로 담은 종목 가운데 추천 10개 안에 들어 있던 비율
- **`import` 한 함수 쓰기** — `recommend_batch.학습하기` 처럼 「파일 이름.함수 이름」으로 스크립트 안의 함수를 노트북에서 부릅니다. 코드를 다시 쓰지 않습니다

**어떻게 동작하나**
1. 스크립트의 `학습하기` 로 **학습 구간만** 써서 학습한다
2. 스크립트의 `추천하기` 로 한 사람에게 (종목 번호, 점수) 10개를 받는다
3. 채점 함수가 원하는 모양(종목 번호 목록)으로 바꾼다
4. 3주차부터 쓰던 `recsys.recall_at_k` 로 채점한다

In [7]:
# 이 셀에서 하는 일 — 스크립트의 함수로 채점용 추천을 만들어 Recall@10 을 잰다
# 왜 하나 — 셀을 파일 하나로 옮겨도 6주차와 같은 추천인지 0.3899 로 확인한다

묶음_채점용 = recommend_batch.학습하기(train, items)                # 학습 구간만으로 학습
성향 = users.set_index("user_id")["persona"].to_dict()             # 딕셔너리 {투자자 번호: 투자성향}
위험도 = items.set_index("item_id")["risk_level"].to_dict()        # 딕셔너리 {종목 번호: 1~5}

def 채점용_추천(사람, 이미):                                        # 채점 함수가 요구하는 모양 (사람, 이미)
    """스크립트의 추천하기가 준 (번호, 점수) 에서 번호만 꺼낸다."""
    번호들 = []
    for 번호, 점수 in recommend_batch.추천하기(묶음_채점용, 사람, 성향, 위험도):
        번호들.append(번호)
    return 번호들

채점_점수, 인원 = recsys.recall_at_k(채점용_추천, train, test)      # 3주차부터 쓰던 채점 함수
print(f"스크립트의 함수로 만든 추천 — Recall@10 = {채점_점수:.4f}   (채점 대상 {인원}명 · 6주차 0.3899)")

스크립트의 함수로 만든 추천 — Recall@10 = 0.3899   (채점 대상 266명 · 6주차 0.3899)


**결과 읽는 법**

- 스크립트 안의 함수로 만든 추천의 Recall@10 이 **0.3899** 입니다.
  - 6주차 위험도 상한 추천과 소수점 넷째 자리까지 같습니다. 채점 대상도 같은 **266명**입니다.
- 셀 여러 개를 파일 하나로 옮겼지만 **추천은 하나도 바뀌지 않았다**는 확인입니다.

**🗣 수업에서 함께 보기**

6주차까지 셀 수십 개로 만들던 추천을 오늘은 **명령 한 줄**로 만들었습니다. 그런데 점수는 **0.3899 로 똑같습니다.**

점수가 그대로라면, 오늘 무엇이 나아진 걸까요? 금융 회사의 입장에서 생각해 보세요.

- 힌트 — 7.1 의 데이터 점검과 7.3 의 「예약 실행」, 그리고 「사람이 셀을 누를 때 생길 수 있는 실수」를 떠올려 보세요.
- 수업에서 함께 이야기합니다. 답은 노트북 맨 아래 **「답」**에도 있습니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📖 7.6 학습과 서빙 — 무거운 일은 한 번만</mark>

> 📖 **혼자 읽기** — 수업에서는 PPT 로 설명하고, 이 절은 복습할 때 직접 돌려 보는 부분입니다.

**7.6 에서 하는 일** — 학습 한 번과 추천 한 번이 각각 얼마나 걸리는지 **시간을 재서** 비교합니다.

**왜 나누나**
- 학습은 무겁고 가끔, 추천 꺼내기는 가볍고 자주 합니다.
- 고객이 앱을 열 때마다 학습을 다시 하면 기다리는 시간이 길어집니다. 그래서 학습은 밤에 한 번, 고객에게는 미리 만든 결과를 꺼내 줍니다.
- 다음 주 API 가 바로 「꺼내 주는」 쪽입니다.

**주의할 점**
- 시간은 컴퓨터마다, 실행할 때마다 조금씩 다릅니다. **정확한 값보다 몇 배 차이인지**를 봅니다.
- `time.perf_counter()` 는 지금 시각을 아주 잘게 재는 시계입니다. 끝난 시각에서 시작 시각을 빼면 걸린 시간입니다.

In [8]:
# 이 셀에서 하는 일 — 학습 한 번과 서빙 300번에 걸린 시간을 잰다
# 왜 하나 — 학습과 서빙을 왜 나누는지 숫자로 확인한다

import time                                               # 시간을 재는 도구

시작 = time.perf_counter()                                # 시작 시각
recommend_batch.학습하기(train, items)                     # 학습 한 번
학습_시간 = time.perf_counter() - 시작                     # 끝난 시각 - 시작 시각

시작 = time.perf_counter()
for 사람 in users["user_id"]:                             # 300명에게 한 번씩
    recommend_batch.추천하기(묶음_채점용, 사람, 성향, 위험도)   # 서빙 — 이미 만든 묶음에서 꺼내기만
서빙_시간 = (time.perf_counter() - 시작) / len(users)       # 한 사람당 평균

print(f"학습 한 번 {학습_시간 * 1000:.1f}ms · 한 사람 추천 {서빙_시간 * 1000:.3f}ms · 약 {학습_시간 / 서빙_시간:,.0f}배")

학습 한 번 46.8ms · 한 사람 추천 0.036ms · 약 1,288배


**결과 읽는 법**

- 첫 숫자는 **학습 한 번**, 둘째 숫자는 **한 사람에게 추천 한 번**에 걸린 시간입니다. 단위 ms 는 1,000분의 1초입니다.
- 마지막 숫자는 학습 한 번이 추천 한 번보다 **몇 배 오래 걸리는지**입니다.
  - 실행할 때마다 조금씩 다르지만, 학습이 **수백 배 이상** 무겁다는 것은 같습니다.
- 우리 데이터는 종목 100개 · 투자자 300명뿐이라 둘 다 빠릅니다. 데이터가 커지면 학습 시간은 훨씬 길어집니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📖 7.7 재현성 — 두 번 돌려도 같은 파일</mark>

> 📖 **혼자 읽기** — 복습할 때 직접 돌려 보는 부분입니다.

**7.7 에서 하는 일** — 스크립트를 **한 번 더** 실행하고, 두 파일이 바이트 하나까지 같은지 비교합니다.

**왜 확인하나**
- 매일 밤 자동으로 돌리는 스크립트가 같은 기록으로 **다른 결과**를 낸다면, 어느 쪽이 맞는지 알 수 없습니다.
- 「어제와 추천이 달라졌다」는 문의가 왔을 때, 데이터가 바뀌어서인지 스크립트가 흔들려서인지 가릴 수 있어야 합니다.
- 행렬분해에는 무작위 요소가 있습니다. `random_state=42` 로 고정해 두었기 때문에 같은 결과가 나와야 합니다. 6주차 FM 의 `seed=42` 와 같은 뜻입니다.

**아래 셀에서 쓰는 도구**
- `shutil.copy` — 파일을 복사합니다.
- `filecmp.cmp(..., shallow=False)` — 두 파일을 **내용까지** 비교합니다. 같으면 True.

In [9]:
# 이 셀에서 하는 일 — 스크립트를 한 번 더 돌리고, 두 결과 파일이 똑같은지 비교한다
# 왜 하나 — 자동으로 매일 돌릴 스크립트는 같은 입력에 같은 결과를 내야 믿을 수 있다

import shutil, filecmp                                     # 파일 복사 도구 · 파일 비교 도구

shutil.copy("output/recommendations.csv", "output/first_run.csv")       # 첫 결과를 옆에 복사해 둔다
!python recommend_batch.py > /dev/null
같은_파일 = filecmp.cmp("output/recommendations.csv", "output/first_run.csv", shallow=False)   # 내용까지 비교
print(f"두 번 실행한 결과 파일이 바이트 하나까지 같은가 : {같은_파일}")

두 번 실행한 결과 파일이 바이트 하나까지 같은가 : True


**결과 읽는 법**

- `True` 는 두 번 실행한 결과 파일이 **바이트 하나까지 같다**는 뜻입니다.
- 같은 거래 기록이면 언제 돌려도 같은 추천이 나옵니다. 이것이 **재현성**입니다.
- 둘째 줄의 `> /dev/null` 은 「스크립트가 찍는 다섯 줄을 화면에 보이지 않게 버려라」라는 뜻입니다. 7.3 에서 이미 봤기 때문입니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📖 7.8 오늘의 마무리</mark>

1. 6주차 최종 추천(행렬분해 + 위험도 상한)을 **다섯 단계 파이프라인**으로 묶었습니다 — 데이터 점검 → 학습 → 일괄 추천 → 규격 검사 → 저장.
2. **데이터 점검** 여섯 가지가 모두 0건이었습니다. 문제가 있으면 학습하기 전에 멈춥니다.
3. 이 모든 일이 `recommend_batch.py` **명령 한 줄**로 끝났고, 결과 파일 3,000줄이 규격 검사를 통과했습니다.
4. 같은 함수로 채점하면 **0.3899** — 6주차와 같습니다. 두 번 돌려도 같은 파일이 나왔습니다(**재현성**).

**오늘 기억할 한 줄** — **추천은 그대로이고, 만드는 데 사람 손이 빠졌습니다.**

다음 주에는 이 결과 파일을 읽어 **추천을 돌려주는 API(FastAPI)** 를 만듭니다. 투자자 번호를 보내면 그 사람의 추천 10개를 돌려주는 창구입니다.

In [10]:
# 이 셀에서 하는 일 — 오늘 점수를 점수판에 남긴다
# 왜 하나 — 점수가 그대로인 주차도 적는다. 무엇이 바뀌었는지 말할 수 있으면 된다

recsys.record(7, "추천 파이프라인 · 실행 한 번", 채점_점수,
              note="데이터 점검 → 학습 → 일괄 추천 → 규격 검사 → 저장을 스크립트 하나로. 점수는 그대로, 사람 손이 빠졌다")

recsys.leaderboard(upto=7)   # 오늘까지 쌓인 점수판 (뒤 주차는 빼고 본다)

레벨 7 · 추천 파이프라인 · 실행 한 번 · Recall@10 = 0.3899


,level,name,recall_at_10,note
0,1,모두에게 같은 인기 순위,0.2163,알고리즘 없음. 인기 상위 10개를 모두에게 같게 추천했다
1,2,취향이 비슷한 세그먼트끼리,0.2684,거래 기록으로 주력 섹터와 평균 위험도를 뽑아 세그먼트로 나눴다
2,3,세그먼트별 목록 — 2주차와 같음,0.2684,추천 방식은 2주차와 같다. 무작위로 나누면 0.3150 이 나오는데 그것은 미래를...
3,4,행렬분해 SVD · 요인 8개,0.3918,규칙을 사람이 정하지 않았다. 사람 285 × 종목 100 표를 요인 8개로 쪼갰다
4,5,2단계 추천 · 후보 생성 + 순위 매기기,0.3382,2주차 세그먼트로 후보 40개를 추리고 그 40개만 4주차 모델로 정렬했다
5,6,투자 성향별 위험도 상한,0.3899,4주차 추천에서 성향 기준을 넘는 종목을 뺐다. 안 맞는 추천 205개 → 0개
6,7,추천 파이프라인 · 실행 한 번,0.3899,데이터 점검 → 학습 → 일괄 추천 → 규격 검사 → 저장을 스크립트 하나로. 점수...


**결과 읽는 법**

- 첫 줄은 오늘 점수를 점수판에 남겼다는 확인입니다. 7주차는 **0.3899** 로 6주차와 같습니다.
- 아래 표는 1주차부터 7주차까지의 점수판입니다.
  - `note` 끝의 `...` 은 글이 길어서 화면에서 잘린 것입니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📝 혼자 해 보기</mark>

수업이 끝난 뒤 혼자 풀어 보는 문제입니다. 문제마다 **어느 절의 결과를 보면 되는지** 제목에 적어 두었습니다.

답은 바로 아래 **「답」**에 있습니다. 먼저 스스로 생각해 본 뒤에 읽어 보세요.

**📝 혼자 해 보기 1 · 7.1 결과**

어느 날 밤 데이터 점검에서 **「똑같은 거래 기록 10건」** 이 나왔다고 해 봅시다. 스크립트는 지금처럼 **멈춰야** 할까요, 아니면 10건만 지우고 **계속 진행해야** 할까요?

- 힌트 — 멈추면 고객은 어제 추천을 그대로 보고, 진행하면 새 추천을 봅니다. 「왜 중복이 생겼는지 모르는 상태」에서 어느 쪽이 더 위험할까요?
- 답은 노트북 맨 아래 **「답」**에 있습니다.

**📝 혼자 해 보기 2 · 7.3 결과**

채점할 때는 **학습 구간만**으로 학습했는데, 스크립트는 **거래 기록 전체**로 학습했습니다.

3주차에서 「미래 기록으로 배우면 데이터 누수」라고 배웠는데, 이건 괜찮을까요? 그리고 두 모델의 추천은 얼마나 다를까요?

- 힌트 — 서비스용 추천은 **언제** 고객에게 보여 주는 것인지 생각해 보세요.
- 답은 노트북 맨 아래 **「답」**에 있습니다.

**📝 혼자 해 보기 3 · 7.4 결과**

④ 규격 검사를 **하지 않고** 파일을 다음 주 API 에 넘긴다고 해 봅시다.

다섯 가지 검사 가운데 하나라도 어긴 파일이 넘어가면, 고객이 받는 추천에는 각각 어떤 일이 생길까요?

- 힌트 — 7.4 의 「왜 중요한가」와 6주차의 적합성원칙을 떠올려 보세요.
- 답은 노트북 맨 아래 **「답」**에 있습니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">답</mark>

먼저 스스로 답을 생각해 본 뒤에 읽어 보세요. 답에 나오는 숫자는 모두 이 노트북과 같은 코드로 실제로 계산한 값입니다.

**함께 보기 답 — 추천은 같지만, 추천을 만드는 일이 믿을 수 있고 · 빠르고 · 사람 없이 돌아가게 되었습니다**

- **사람의 실수가 빠졌습니다**
  - 셀을 누르다 보면 순서를 틀리거나 한 셀을 빼먹을 수 있습니다. 스크립트는 늘 같은 순서로 돕니다.
- **매일 자동으로 돌릴 수 있습니다**
  - 명령 한 줄이라 새벽에 예약 실행을 걸어 둘 수 있습니다. 고객은 아침마다 어제 거래까지 반영된 추천을 받습니다.
- **잘못된 추천이 나가지 않게 막습니다**
  - 들어오는 데이터(데이터 점검)와 나가는 파일(규격 검사)을 코드가 검사하고, 문제가 있으면 멈춥니다.
  - 특히 위험도 상한을 넘는 추천이 고객에게 나가지 않습니다 — 적합성원칙(6주차).
- **다음 단계와 맞물립니다**
  - 규격이 정해진 파일 하나가 남아서, 다음 주 API 가 그대로 읽어 쓸 수 있습니다.
- **한 줄로 정리하면** — 오늘은 점수가 아니라 **운영**이 좋아졌습니다. 실제 서비스에서 추천 모델의 가치는 점수와 운영이 함께 갖춰질 때 나옵니다.

**혼자 해 보기 1 답 — 멈추는 쪽이 안전합니다. 원인을 모르는 채로 고치면 다른 문제를 덮을 수 있기 때문입니다**

- 중복 10건은 결과일 뿐, **왜 생겼는지**는 모릅니다.
  - 전송이 두 번 된 것일 수도 있고, 거래 시스템 자체에 오류가 생긴 것일 수도 있습니다.
  - 후자라면 중복 말고도 깨진 기록이 더 있을 수 있습니다. 10건만 지우고 진행하면 그 문제를 덮게 됩니다.
- 멈춰도 **큰 손해가 없습니다.** 고객은 어제 만든 추천을 하루 더 봅니다.
- 반대로 깨진 데이터로 진행하면 **조용히 틀린 추천**이 3,000개 나갈 수 있습니다. 금융에서는 성향에 맞지 않는 추천으로 이어질 수도 있습니다.
- 실무에서는 「멈추고 담당자에게 알림 → 원인 확인 → 다시 실행」이 기본입니다. 원인이 확실히 알려진 경우에만 「자동으로 지우고 진행」 규칙을 따로 만듭니다.
- **한 줄로 정리하면** — 확신이 없으면 멈춘다. 하루 늦은 추천이 틀린 추천보다 낫습니다.

**혼자 해 보기 2 답 — 괜찮습니다. 서비스용 추천을 보여 주는 시점에는 그 기록이 모두 과거이기 때문입니다**

- 데이터 누수는 **채점할 때** 생기는 문제입니다.
  - 채점은 「모르는 미래를 얼마나 맞히나」를 재는 일이라, 맞힐 기간을 학습에 넣으면 실력이 부풀려집니다(3주차).
- 서비스용 추천은 **내일 고객에게 보여 줄** 추천입니다.
  - 그 시점에는 마지막 한 달도 이미 지난 과거입니다. 가장 최근 기록까지 모두 쓰는 것이 맞습니다.
  - 그래서 실무에서는 **채점용 모델로 실력을 재고, 서비스용 모델은 전체 기간으로 다시 학습**합니다.
- 두 모델의 추천은 꽤 비슷하지만 같지는 않습니다.
  - 300명의 추천 10개를 비교하면 평균 **7.45개**가 같고, 10개가 모두 같은 사람은 **11명**입니다.
  - 채점용에 없던 **신규 15명**도 서비스용에서는 자기 기록으로 추천을 받습니다.
- **한 줄로 정리하면** — 실력은 미래를 숨기고 재고, 서비스는 아는 것을 모두 써서 만듭니다.

**혼자 해 보기 3 답 — API 가 멈추거나, 추천이 비거나, 고객에게 맞지 않는 종목이 나갑니다**

- **열 이름 · 순서가 다르면** — API 가 `item_name` 같은 열을 찾지 못해 멈춥니다. 그 API 를 쓰는 화면도 함께 멈춥니다.
- **추천이 없는 사람이 있으면** — API 가 그 고객에게 돌려줄 추천이 없습니다.
- **10줄이 아닌 사람이 있으면** — 고객마다 받는 추천 개수가 달라집니다. 10개를 기대한 화면은 칸이 어긋납니다.
- **상한을 넘는 종목이 있으면** — 안정형 고객에게 위험도 4 \~ 5 종목이 그대로 보입니다.
  - 6주차에서 본 적합성원칙에 어긋나는 추천이 고객에게 나가는 것입니다.
- **이미 담은 종목이 있으면** — 고객이 이미 가진 종목을 다시 추천해서 추천이 쓸모없어집니다.
- **한 줄로 정리하면** — 규격 검사는 잘못된 파일이 API 와 고객에게 가지 않게 막는 마지막 확인입니다.